In [ ]:
# Inspired by https://medium.com/@soyoungluna/tutorial-feature-engineering-for-weekly-time-series-forecasting-in-pyspark-b207c41869f4

In [1]:
import findspark
findspark.init("/opt/spark")

In [123]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *
import datetime
from datetime import timedelta, date
import holidays
from pyspark.sql.window import Window

In [3]:
spark = SparkSession.builder.appName("Time Series").master("local[*]") \
.config("spark.driver.memory", "6g") \
.getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/07/29 08:00:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
sc = spark.sparkContext

In [5]:
from pyspark import SparkFiles
p1 ="https://github.com/erkansirin78/datasets/raw/master/OnlineRetail2.csv"
sc.addFile(p1)
df = spark.read.csv("file:///"+SparkFiles.get("*"), header=True, inferSchema=True)

In [6]:
df.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)



In [7]:
df.show(5)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|     2.55|   17850.0|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|     2.75|   17850.0|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
only showing top 5 rows



In [26]:
df.createTempView("retail")

In [38]:
spark.sql("""

select Country, COUNT(*) 
from retail
GROUP BY Country
ORDER BY 2 DESC

""").limit(100).toPandas()

,Country,count(1)
0,United Kingdom,495478
1,Germany,9495
2,France,8557
3,EIRE,8196
4,Spain,2533
5,Netherlands,2371
6,Belgium,2069
7,Switzerland,2002
8,Portugal,1519
9,Australia,1259


# Calculate revenue

In [103]:
df2 = df.withColumn("revenue", df["Quantity"] * df["UnitPrice"]) \
.filter("Country NOT IN ('EIRE', 'Unspecified', 'Hong Kong', 'Channel Islands', 'European Community', 'United Arab Emirates', 'Czech Republic', 'Lebanon', 'Saudi Arabia', 'RSA')")

In [104]:
df2.limit(5).toPandas()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


# Add holiday flag

In [105]:
@F.udf(BooleanType())
def is_holiday(country_name, date_str):
    date_str = str(date_str)
    try:
        # Special handling for United Kingdom
        if country_name.lower() == "united kingdom":
            from workalendar.europe import UnitedKingdom
            cal = UnitedKingdom()
            date_obj = date.fromisoformat(date_str)
            return cal.is_holiday(date_obj)

        # Get country code
        country_code = holidays.country_holidays(country_name).country

        # Check if date is a holiday
        country_holidays = holidays.CountryHoliday(country_code) 
        date_obj = date.fromisoformat(date_str)
        return date_obj in country_holidays

    except (ValueError, KeyError):  # Handle invalid date or country
        return False

In [106]:
df3 = df2.withColumn("is_holiday", is_holiday(F.col('Country'), F.to_date('InvoiceDate')))

In [107]:
df3.limit(5).toPandas()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,revenue,is_holiday
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,False
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,False
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,False
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,False
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,False


# Aggregate features at country & week level

In [118]:
agg_df = df3.withColumn("week_ending_date", F.expr("date_add(InvoiceDate, 7 - dayofweek(InvoiceDate))")) \
            .groupBy("Country", "week_ending_date") \
            .agg(F.countDistinct("InvoiceNo").alias("num_of_invoices"),
                 F.countDistinct("StockCode").alias("num_of_distinct_items"),
                 F.sum("Quantity").alias("quantity"),
                 F.countDistinct("CustomerID").alias("num_of_distinct_customers"),
                 F.avg("UnitPrice").alias("avg_unit_price"),
                 F.median("UnitPrice").alias("median_unit_price"),
                 F.sum("revenue").alias("revenue"),
                 F.countDistinct("is_holiday").alias("count_dist_holiday"))

In [119]:
agg_df.cache()

24/07/29 08:32:14 WARN CacheManager: Asked to cache already cached data.


DataFrame[Country: string, week_ending_date: date, num_of_invoices: bigint, num_of_distinct_items: bigint, quantity: bigint, num_of_distinct_customers: bigint, avg_unit_price: double, median_unit_price: double, revenue: double, count_dist_holiday: bigint]

In [120]:
agg_df.limit(5).toPandas()

,Country,week_ending_date,num_of_invoices,num_of_distinct_items,quantity,num_of_distinct_customers,avg_unit_price,median_unit_price,revenue,count_dist_holiday
0,Spain,2011-01-08,2,99,629,2,14.269091,1.95,12.67,2
1,Spain,2011-03-26,3,104,1193,3,3.744375,1.65,2854.90,1
2,USA,2011-12-10,2,32,212,2,1.764375,1.67,345.32,1
3,United Kingdom,2011-05-28,488,1740,76583,313,3.869271,2.10,152768.66,1
4,Germany,2011-10-08,20,305,6670,17,3.848256,1.65,10954.50,1


# Add Time-Related Features

In [121]:
agg_df = agg_df.withColumn("year", F.year("week_ending_date"))
agg_df = agg_df.withColumn("month", F.month("week_ending_date"))
agg_df = agg_df.withColumn("week_number", F.weekofyear("week_ending_date"))

In [122]:
agg_df.limit(5).toPandas()

,Country,week_ending_date,num_of_invoices,num_of_distinct_items,quantity,num_of_distinct_customers,avg_unit_price,median_unit_price,revenue,count_dist_holiday,year,month,week_number
0,Spain,2011-01-08,2,99,629,2,14.269091,1.95,12.67,2,2011,1,1
1,Spain,2011-03-26,3,104,1193,3,3.744375,1.65,2854.90,1,2011,3,12
2,USA,2011-12-10,2,32,212,2,1.764375,1.67,345.32,1,2011,12,49
3,United Kingdom,2011-05-28,488,1740,76583,313,3.869271,2.10,152768.66,1,2011,5,21
4,Germany,2011-10-08,20,305,6670,17,3.848256,1.65,10954.50,1,2011,10,40


# Add Lag Features

In [124]:
# Function to add lag features
def add_lag_features(df, metrics, lags):
    windowSpec = Window.partitionBy("Country").orderBy("week_ending_date")
    for lag_period in lags:
        for metric in metrics:
            lag_col_name = f"{metric}_lag{lag_period}"
            df = df.withColumn(lag_col_name, F.lag(F.col(metric), lag_period).over(windowSpec))
    return df

# Define the metrics and lag periods
metrics = ["num_of_invoices", "num_of_distinct_items", "quantity", "num_of_distinct_customers", "avg_unit_price", "median_unit_price", "revenue", "count_dist_holiday"]
lags = [1, 2, 4]

# Add lag features to the DataFrame
agg_lag_df = add_lag_features(agg_df, metrics, lags)

# Select columns containing 'lag' in their names
lag_columns = [col for col in agg_lag_df.columns if 'lag' in col]

# Display DataFrame with selected columns
agg_lag_df.select(lag_columns).limit(10).toPandas()

,num_of_invoices_lag1,num_of_distinct_items_lag1,quantity_lag1,num_of_distinct_customers_lag1,avg_unit_price_lag1,median_unit_price_lag1,revenue_lag1,count_dist_holiday_lag1,num_of_invoices_lag2,num_of_distinct_items_lag2,...,revenue_lag2,count_dist_holiday_lag2,num_of_invoices_lag4,num_of_distinct_items_lag4,quantity_lag4,num_of_distinct_customers_lag4,avg_unit_price_lag4,median_unit_price_lag4,revenue_lag4,count_dist_holiday_lag4
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.0,14.0,107.0,1.0,5.278571,5.700,358.25,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1.0,8.0,214.0,1.0,2.726250,1.650,258.90,1.0,1.0,14.0,...,358.25,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2.0,12.0,133.0,2.0,3.688462,1.950,387.95,1.0,1.0,8.0,...,258.90,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2.0,48.0,4802.0,2.0,1.837083,1.450,7154.38,1.0,2.0,12.0,...,387.95,1.0,1.0,14.0,107.0,1.0,5.278571,5.700,358.25,1.0
5,3.0,23.0,394.0,2.0,2.223913,2.250,721.10,1.0,2.0,48.0,...,7154.38,1.0,1.0,8.0,214.0,1.0,2.726250,1.650,258.90,1.0
6,3.0,35.0,338.0,3.0,2.995833,2.525,757.55,1.0,3.0,23.0,...,721.10,1.0,2.0,12.0,133.0,2.0,3.688462,1.950,387.95,1.0
7,1.0,20.0,110.0,1.0,4.588000,3.750,384.68,1.0,3.0,35.0,...,757.55,1.0,2.0,48.0,4802.0,2.0,1.837083,1.450,7154.38,1.0
8,4.0,7.0,58.0,2.0,3.958333,4.600,192.00,1.0,1.0,20.0,...,384.68,1.0,3.0,23.0,394.0,2.0,2.223913,2.250,721.10,1.0
9,1.0,69.0,8384.0,1.0,2.778696,1.690,14022.92,1.0,4.0,7.0,...,192.00,1.0,3.0,35.0,338.0,3.0,2.995833,2.525,757.55,1.0


# Add Window Average Features

In [126]:
# Function to add window average features
def add_window_avg_features(df, metrics, window_sizes):
    for metric in metrics:
        for window_size in window_sizes:
            window_col_name = f"avg_{metric}_l{window_size}w"
            windowSpec = Window.partitionBy("Country").orderBy("week_ending_date").rowsBetween(-window_size, 0)
            df = df.withColumn(window_col_name, F.avg(F.col(metric)).over(windowSpec))
    return df

# Define the metrics and window sizes
metrics = ["num_of_invoices", "num_of_distinct_items", "quantity", "num_of_distinct_customers", "revenue"]
window_sizes = [1, 4, 12,52]

# Add window average features to the DataFrame
aggwin_avg_features_df = add_window_avg_features(agg_lag_df, metrics, window_sizes)

In [127]:
aggwin_avg_features_df.limit(10).toPandas()

24/07/29 08:37:25 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,Country,week_ending_date,num_of_invoices,num_of_distinct_items,quantity,num_of_distinct_customers,avg_unit_price,median_unit_price,revenue,count_dist_holiday,...,avg_quantity_l12w,avg_quantity_l52w,avg_num_of_distinct_customers_l1w,avg_num_of_distinct_customers_l4w,avg_num_of_distinct_customers_l12w,avg_num_of_distinct_customers_l52w,avg_revenue_l1w,avg_revenue_l4w,avg_revenue_l12w,avg_revenue_l52w
0,Australia,2010-12-04,1,14,107,1,5.278571,5.700,358.25,1,...,107.000000,107.000000,1.0,1.000000,1.000000,1.000000,358.250,358.250000,358.250000,358.250000
1,Australia,2010-12-11,1,8,214,1,2.726250,1.650,258.90,1,...,160.500000,160.500000,1.0,1.000000,1.000000,1.000000,308.575,308.575000,308.575000,308.575000
2,Australia,2010-12-18,2,12,133,2,3.688462,1.950,387.95,1,...,151.333333,151.333333,1.5,1.333333,1.333333,1.333333,323.425,335.033333,335.033333,335.033333
3,Australia,2011-01-08,2,48,4802,2,1.837083,1.450,7154.38,1,...,1314.000000,1314.000000,2.0,1.500000,1.500000,1.500000,3771.165,2039.870000,2039.870000,2039.870000
4,Australia,2011-01-15,3,23,394,2,2.223913,2.250,721.10,1,...,1130.000000,1130.000000,2.0,1.600000,1.600000,1.600000,3937.740,1776.116000,1776.116000,1776.116000
5,Australia,2011-01-22,3,35,338,3,2.995833,2.525,757.55,1,...,998.000000,998.000000,2.5,2.000000,1.833333,1.833333,739.325,1855.976000,1606.355000,1606.355000
6,Australia,2011-01-29,1,20,110,1,4.588000,3.750,384.68,1,...,871.142857,871.142857,2.0,2.000000,1.714286,1.714286,571.115,1881.132000,1431.830000,1431.830000
7,Australia,2011-02-12,4,7,58,2,3.958333,4.600,192.00,1,...,769.500000,769.500000,1.5,2.000000,1.750000,1.750000,288.340,1841.942000,1276.851250,1276.851250
8,Australia,2011-02-19,1,69,8384,1,2.778696,1.690,14022.92,1,...,1615.555556,1615.555556,1.5,1.800000,1.666667,1.666667,7107.460,3215.650000,2693.081111,2693.081111
9,Australia,2011-03-05,3,92,10379,2,3.148617,1.650,16970.69,1,...,2491.900000,2491.900000,1.5,1.800000,1.700000,1.700000,15496.805,6465.568000,4120.842000,4120.842000


# Add Moving Average Features (EWMA)

In [128]:
def calculate_ewma_lag1(df, metric, alpha):
    """
    Calculate the exponential weighted moving average (EWMA) for a given metric with a specified alpha.

    Args:
    - df: A DataFrame containing the data.
    - metric: The name of the column containing the metric.
    - alpha: The smoothing factor (0 < alpha <= 1).

    Returns:
    - df: The DataFrame with a new column containing the EWMA values.
    """
    # Define a window specification partitioned by the desired columns
    windowSpec = Window.partitionBy("Country").orderBy("week_ending_date")
    
    # Calculate the EWMA using the lag 1 value
    ewma_column = f"{metric}_ewma_lag1_{str(alpha).replace('.', '')}"
    df = df.withColumn(ewma_column,
                       F.lag(F.col(metric)).over(windowSpec) * (1 - alpha) +
                       F.col(metric) * alpha)

    
    return df

# Chose 0.2, 0.5 and 0.8 smoothing factors to start with
alpha_values = [0.2, 0.5, 0.8]
metrics = ["quantity", "revenue"]

for alpha in alpha_values:
    for metric in metrics:
        aggwin_avg_features_df = calculate_ewma_lag1(aggwin_avg_features_df, metric, alpha)

In [129]:
aggwin_avg_features_df.limit(10).toPandas()

,Country,week_ending_date,num_of_invoices,num_of_distinct_items,quantity,num_of_distinct_customers,avg_unit_price,median_unit_price,revenue,count_dist_holiday,...,avg_revenue_l1w,avg_revenue_l4w,avg_revenue_l12w,avg_revenue_l52w,quantity_ewma_lag1_02,revenue_ewma_lag1_02,quantity_ewma_lag1_05,revenue_ewma_lag1_05,quantity_ewma_lag1_08,revenue_ewma_lag1_08
0,Australia,2010-12-04,1,14,107,1,5.278571,5.700,358.25,1,...,358.250,358.250000,358.250000,358.250000,NaN,NaN,NaN,NaN,NaN,NaN
1,Australia,2010-12-11,1,8,214,1,2.726250,1.650,258.90,1,...,308.575,308.575000,308.575000,308.575000,128.4,338.380,160.5,308.575,192.6,278.770
2,Australia,2010-12-18,2,12,133,2,3.688462,1.950,387.95,1,...,323.425,335.033333,335.033333,335.033333,197.8,284.710,173.5,323.425,149.2,362.140
3,Australia,2011-01-08,2,48,4802,2,1.837083,1.450,7154.38,1,...,3771.165,2039.870000,2039.870000,2039.870000,1066.8,1741.236,2467.5,3771.165,3868.2,5801.094
4,Australia,2011-01-15,3,23,394,2,2.223913,2.250,721.10,1,...,3937.740,1776.116000,1776.116000,1776.116000,3920.4,5867.724,2598.0,3937.740,1275.6,2007.756
5,Australia,2011-01-22,3,35,338,3,2.995833,2.525,757.55,1,...,739.325,1855.976000,1606.355000,1606.355000,382.8,728.390,366.0,739.325,349.2,750.260
6,Australia,2011-01-29,1,20,110,1,4.588000,3.750,384.68,1,...,571.115,1881.132000,1431.830000,1431.830000,292.4,682.976,224.0,571.115,155.6,459.254
7,Australia,2011-02-12,4,7,58,2,3.958333,4.600,192.00,1,...,288.340,1841.942000,1276.851250,1276.851250,99.6,346.144,84.0,288.340,68.4,230.536
8,Australia,2011-02-19,1,69,8384,1,2.778696,1.690,14022.92,1,...,7107.460,3215.650000,2693.081111,2693.081111,1723.2,2958.184,4221.0,7107.460,6718.8,11256.736
9,Australia,2011-03-05,3,92,10379,2,3.148617,1.650,16970.69,1,...,15496.805,6465.568000,4120.842000,4120.842000,8783.0,14612.474,9381.5,15496.805,9980.0,16381.136


# Add Delta Features

In [131]:
def add_delta_features_lag1(df, metrics):
    windowSpec = Window.partitionBy("Country").orderBy("week_ending_date")
    for metric in metrics:
        delta_col_name = f"delta_{metric}_lag1"
        lag_col_name = f"{metric}_lag1"
        df = df.withColumn(delta_col_name, F.col(metric) - F.lag(F.col(metric), 2).over(windowSpec))
    return df

# Define the metrics for which delta features will be calculated
metrics = ["num_of_invoices", "num_of_distinct_items", "quantity", "num_of_distinct_customers", "avg_unit_price", "median_unit_price", "revenue"]

# Add delta features to the DataFrame
aggwin_avg_features_df = add_delta_features_lag1(aggwin_avg_features_df, metrics)

In [132]:
aggwin_avg_features_df.limit(10).toPandas()

,Country,week_ending_date,num_of_invoices,num_of_distinct_items,quantity,num_of_distinct_customers,avg_unit_price,median_unit_price,revenue,count_dist_holiday,...,revenue_ewma_lag1_05,quantity_ewma_lag1_08,revenue_ewma_lag1_08,delta_num_of_invoices_lag1,delta_num_of_distinct_items_lag1,delta_quantity_lag1,delta_num_of_distinct_customers_lag1,delta_avg_unit_price_lag1,delta_median_unit_price_lag1,delta_revenue_lag1
0,Australia,2010-12-04,1,14,107,1,5.278571,5.700,358.25,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Australia,2010-12-11,1,8,214,1,2.726250,1.650,258.90,1,...,308.575,192.6,278.770,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Australia,2010-12-18,2,12,133,2,3.688462,1.950,387.95,1,...,323.425,149.2,362.140,1.0,-2.0,26.0,1.0,-1.590110,-3.750,29.70
3,Australia,2011-01-08,2,48,4802,2,1.837083,1.450,7154.38,1,...,3771.165,3868.2,5801.094,1.0,40.0,4588.0,1.0,-0.889167,-0.200,6895.48
4,Australia,2011-01-15,3,23,394,2,2.223913,2.250,721.10,1,...,3937.740,1275.6,2007.756,1.0,11.0,261.0,0.0,-1.464548,0.300,333.15
5,Australia,2011-01-22,3,35,338,3,2.995833,2.525,757.55,1,...,739.325,349.2,750.260,1.0,-13.0,-4464.0,1.0,1.158750,1.075,-6396.83
6,Australia,2011-01-29,1,20,110,1,4.588000,3.750,384.68,1,...,571.115,155.6,459.254,-2.0,-3.0,-284.0,-1.0,2.364087,1.500,-336.42
7,Australia,2011-02-12,4,7,58,2,3.958333,4.600,192.00,1,...,288.340,68.4,230.536,1.0,-28.0,-280.0,-1.0,0.962500,2.075,-565.55
8,Australia,2011-02-19,1,69,8384,1,2.778696,1.690,14022.92,1,...,7107.460,6718.8,11256.736,0.0,49.0,8274.0,0.0,-1.809304,-2.060,13638.24
9,Australia,2011-03-05,3,92,10379,2,3.148617,1.650,16970.69,1,...,15496.805,9980.0,16381.136,-1.0,85.0,10321.0,0.0,-0.809716,-2.950,16778.69
